# Observation Walkthrough

A living reference for the `agent(obs_dict) -> list[int]` contract: run one
random-vs-random match locally via `cg.game` and inspect the shape of
`Observation` / `SelectData` / `Option` as the engine actually returns them.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import cg_bridge  # noqa: F401 (adds the vendored cg SDK to sys.path)
from cg_bridge import battle_finish, battle_select, battle_start, to_observation_class, EnergyType
from deck import build_basic_mono_deck, load_card_pool

pool = load_card_pool()
deck = build_basic_mono_deck(pool, EnergyType.FIRE)

obs_dict, start_data = battle_start(deck, deck)
print("Top-level obs_dict keys:", sorted(obs_dict.keys()))

Top-level obs_dict keys: ['current', 'logs', 'search_begin_input', 'select']


In [2]:
obs = to_observation_class(obs_dict)
print("select.type:", obs.select.type)
print("select.context:", obs.select.context)
print("minCount/maxCount:", obs.select.minCount, obs.select.maxCount)
print("n options:", len(obs.select.option))
print("option types offered:", sorted({o.type for o in obs.select.option}))
print("current.turn / yourIndex / result:", obs.current.turn, obs.current.yourIndex, obs.current.result)

select.type: 9
select.context: 41
minCount/maxCount: 1 1
n options: 2
option types offered: [1, 2]
current.turn / yourIndex / result: 0 0 -1


In [3]:
# Step through a handful of selections, collecting the distinct SelectContext
# values encountered, as a reference for what an agent might see early in a match.
from agents.random_agent import agent as random_agent

contexts_seen = set()
obs_dict_i = obs_dict
for _ in range(30):
    obs_i = to_observation_class(obs_dict_i)
    if obs_i.current.result != -1:
        break
    contexts_seen.add(obs_i.select.context)
    selection = random_agent(obs_dict_i)
    obs_dict_i = battle_select(selection)

battle_finish()
print("SelectContext values seen in the first 30 actions:", sorted(contexts_seen))

SelectContext values seen in the first 30 actions: [0, 1, 2, 41]
